In [ ]:
#Dependencies
import mne
from mne.preprocessing import ICA
from mne_icalabel import label_components

import pathlib as pth
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt

In [ ]:
#Load data
data_dir = pth.Path(r"C:\Users\Isha Paranjape\OneDrive\Desktop\ISA BCI Project\Exoskeleton Literature\bci-project\raw_data\P01") # Change to new data directory
raw = mne.io.read_raw_edf(data_dir / "P01_NP_S001.edf", preload=False) # Each participant will go twice since they have 2 sessions
raw.resample(250)  # Downsample for faster processing, adjust as needed for new data and computational resources

raw.load_data()

In [ ]:
#Change to code to plot PSD and check data quality
raw.compute_psd(fmin=1, fmax=60).plot(picks = "eeg", average = False, show = True) # Check for line noise and bad channels, adjust freqs and plot settings as needed
plt.show(block = True)

In [ ]:
#Rename channels to match montage naming convention
print(raw.info['ch_names']) # Check original channel names before renaming

clean_name = {ch: ch.replace('EEG ', '').replace('-5Z', '') for ch in raw.ch_names}
raw.rename_channels(clean_name)
print(raw.info['ch_names']) # Check channel names after renaming

new_mapping = {
    '1Z': 'Fpz', '2Z': 'AFz', '3Z': 'Fz', '4Z': 'FCz', '6Z': 'CPz', '7Z': 'Pz', '8Z': 'POz', '9Z': 'Oz',
    '1L': 'Fp1', '2L': 'AF3', '3L': 'F1', '4L': 'FC1', '5L': 'C1', '6L': 'CP1', '7L': 'P1', '8L': 'PO3', '9L': 'O1',
    '1R': 'Fp2', '2R': 'AF4', '3R': 'F2', '4R': 'FC2', '5R': 'C2', '6R': 'CP2', '7R': 'P2', '8R': 'PO4', '9R': 'O2',
    '1LA': 'F3', '2LA': 'C3', '3LA': 'P3',
    '1LB': 'AF7', '2LB': 'F5', '3LB': 'FC5', '4LB': 'CP5', '5LB': 'P5',
    '1LC': 'F7', '2LC': 'FT7', '3LC': 'T7', '4LC': 'TP7', '5LC': 'PO7',
    '1LD': 'F9', '2LD': 'FT9', '3LD': 'TP9', '4LD': 'P9',
    '1RA': 'F4', '2RA': 'C4', '3RA': 'P4',
    '1RB': 'AF8', '2RB': 'F6', '3RB': 'FC6', '4RB': 'CP6', '5RB': 'P6',
    '1RC': 'F8', '2RC': 'FT8', '3RC': 'T8', '4RC': 'TP8', '5RC': 'PO8',
    '1RD': 'F10', '2RD': 'FT10', '3RD': 'TP10', '4RD': 'P10'
}
raw.rename_channels(new_mapping)

print(raw.info['ch_names']) # Check channel names after renaming


In [ ]:
# Plot data to inspect and drop bad channels
raw.plot(start=90, duration=15, block = True, bad_color = 'red', scalings='auto') #Avoid hardcording bad channels
# For P01: raw.info['bads'] += ['10R', '11R', '10L', '11L', 'F7', 'TP7', 'PO7', 'P9', 'FT10', 'TP10'] 
print(raw.info['bads'])
raw.drop_channels(raw.info['bads']) # Drop bad channels after inspection

#Montage and re-reference to fit and normalize the data
montage = mne.channels.make_standard_montage('brainproducts-RNP-BA-128') # Works just fine for our cap, ICA doesnt require anatomically exact montage
raw.set_montage(montage, on_missing = 'ignore', match_case=False)
raw.set_eeg_reference('average', projection = False) # CAR referencing, adjust as needed for new data and analysis goals
raw.plot() # Check data after referencing

#Plot to verify montage and referencing
raw.plot_sensors(kind='topomap', show_names = True, block = True) # Check montage

In [ ]:
# Make unfiltered copy for ICA
raw_unf = raw.copy()

# ICA to remove artifacts
raw_unf.filter(l_freq=1.0, h_freq=100.0)              # Wideband filter for ICA separation. Usually 1-100 Hz but recorded data was filtered at 70 Hz
raw_unf.notch_filter(freqs=50.0)                    # Notch filter to remove line noise

# Fit ICA using EXTENDED Infomax (ICLabel-compatible)
ica = ICA(
    n_components=0.99, # Retain components explaining 99% of variance (adjust as needed)
    method='infomax', 
    fit_params=dict(extended=True),                        
    random_state=97,
    max_iter='auto' # Let MNE choose the number of iterations based on convergence criteria
)

ica.fit(raw_unf)                                      # Fit on wideband data
print(f"Fitted {ica.n_components_} components") # 

# ICLabel classification
iclabels_dict = label_components(
    raw_unf,                                       
    ica, 
    method="iclabel" 
)

labels = iclabels_dict["labels"] # Main ICLabel labels (e.g., 'brain', 'eye blink', 'muscle', etc.)
probs = iclabels_dict["y_pred_proba"] # Class probabilities for each IC and class (shape: n_components x n_classes)

print("IC labels:", labels)
ica.plot_properties(raw, picks=[0, 12], verbose=False) # Apply to data for better visualization of properties, adjust picks as needed

# Exclude components classified as artifacts
ica.exclude = [i for i, lab in enumerate(labels) if lab not in ['brain', 'other']] # Exclude non-brain components, adjust as needed based on ICLabel results and visual inspection
print(f"Excluding {len(ica.exclude)} ICs: {ica.exclude}")

# Safe plotting
if ica.exclude:
    ica.plot_properties(raw, picks=ica.exclude[:4], show=True) # Plot properties of the first few excluded components for verification, adjust picks as needed for new data
else:
    print("No auto-exclusions, inspect manually")
ica.plot_components(show=True)

#Filter main to remove unnecessary frequencies
raw.filter(l_freq=0.5, h_freq=40.0) # Later crop to 8-30 Hz for mu/beta focus?
raw.notch_filter(freqs=50.0)

# Apply ICA to original data
raw_clean = ica.apply(raw.copy())

In [ ]:
#Event extraction
#Keep only PsychoPy TriggerStream events (ignore impedance + 0)
trigs = raw.annotations 

print(type(trigs))
print(trigs[0:100]) 

In [ ]:
# Extract useful trigger events
good_idx = [
    i for i, desc in enumerate(trigs.description)
    if ('TriggerStream' in desc) 
    and (not desc.startswith('0')) 
]

print(good_idx[0:20])

ons_good = np.array(trigs.onset)[good_idx]
durs_good = np.array(trigs.duration)[good_idx]
descs_good = np.array(trigs.description)[good_idx]

print(ons_good[0:20])
print(durs_good[0:20])
print(descs_good[0:20])


In [ ]:
# eego records 2 triggers for an event, need to figure out how to remove duplicates without losing events,
# this is a workaround to keep only the first occurrence of each event type
keep = [0]  # always keep first annotation

for i in range(1, len(descs_good)):
    if not (descs_good[i] == descs_good[i-1] and (ons_good[i] - ons_good[i-1]) < 1.0): # If the same event type occurs within 1 second, consider it a duplicate
        keep.append(i)

keep.pop(-1)

clean_annotations = mne.Annotations(
    onset=ons_good[keep],
    duration=durs_good[keep],
    description=descs_good[keep],
)

#Verify cleaned annotations
for i in range(len(clean_annotations)):
    print(
        clean_annotations.onset[i],
        clean_annotations.description[i])



In [ ]:
# Set the cleaned annotations back to raw data
raw_clean.set_annotations(clean_annotations)

In [ ]:
# Clean up event descriptions to be just the trigger number for easier event extraction
raw_clean.annotations.rename({
    '101###TriggerStream 101': 101,
    '102###TriggerStream 102': 102,
    '201###TriggerStream 201': 201,
    '202###TriggerStream 202': 202,
    '301###TriggerStream 301': 301,
    '302###TriggerStream 302': 302
})

# Create class relevant event IDs
class_event_ids = {
    '101': 1,  # Class 1
    '102': 2,  # Class 2
    '201': 3,  # Class 3
    '202': 4,  # Class 4
    '301': 5,  # Class 5
    '302': 6,  # Class 6
}

events, event_id = mne.events_from_annotations(raw_clean, event_id=class_event_ids)
print("Event IDs:", event_id)

In [ ]:
#Epoching
#Make a copy of raw data to preserve original
raw_copy = raw_clean.copy()
raw_copy.filter(l_freq=8.0, h_freq=30.0) # Filter the copy for mu and beta focus

#Epochs (this will be the same for all classes for the new data)
tfr_epochs = mne.Epochs(
    raw_copy,
    events,
    event_id = event_id,
    tmin = -2.0,
    tmax = 13.0,
    baseline = None,  # Pre-cue baseline
    preload = True,
    event_repeated='drop'
)
print(tfr_epochs)  # Verify separated epochs by class

alg_epochs = tfr_epochs.copy().crop(tmin=1.0, tmax=8.0) #Crop for classifier algorithm
print(alg_epochs) # Verify cropped epochs for algorithm


In [ ]:
# Save epochs
tfr_epochs.save("P01_epochs-epo.fif", overwrite=True)
alg_epochs.save("P01_alg_epochs-epo.fif", overwrite=True)